In [ ]:
import scanpy as sc
import sys
import os

sys.path.append("../../analyses/utils/HECA/")
import data_cleanup
import mapping

In [ ]:
# %load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
testrun = False

In [ ]:
batch_covariate = "sample"
kl = 1

In [ ]:
suffix = ""
if testrun:
    suffix = "_test"
path_reference = "../../../../../workspace/lisa.sikkema/mapqc/data/HECA/reference/endometriumAtlasV2_cells_with_counts.h5ad"
path_query = "../../../../../workspace/lisa.sikkema/mapqc/data/HECA/queries_counts/Asherman_syndrome_wo_wang.h5ad"
path_ref_model_dir = "./scanvi_ref_model/"
path_surgery_model_dir = (
    f"../../results/HECA/batch_{batch_covariate}_kl{kl}/surgery_model_and_output_emb"
)
path_to_mapped_query_embedding = os.path.join(
    path_surgery_model_dir, f"query_emb{suffix}.h5ad"
)
path_to_joint_ref_q_counts_object = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HECA/HECA_Santamaria{suffix}.h5ad"
path_to_joint_ref_q_latent_object = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HECA/HECA_Santamaria_batch_{batch_covariate}_kl{kl}_latent{suffix}.h5ad"

In [ ]:
adata_ref = data_cleanup.import_and_prepare_reference(path_reference, testrun=testrun)

In [ ]:
adata_q = data_cleanup.import_and_prepare_query(path_query, testrun=testrun)

In [ ]:
adata_q.obs.Group.value_counts()

In [ ]:
adata_q = mapping.prepare_query_data_for_mapping(adata_q, batch_key="sample")

In [ ]:
(
    adata_ref_latent,
    adata_q_subset,
) = mapping.subset_query_data_for_mapping_and_get_ref_latent(
    adata_ref=adata_ref, adata_q=adata_q, path_to_ref_model_dir=path_ref_model_dir
)

In [ ]:
mapping.map_data_to_HECA_reference(
    prepped_query_adata=adata_q_subset,
    path_ref_model_dir=path_ref_model_dir,
    path_surgery_model_dir=path_surgery_model_dir,
    path_output_query_emb=path_to_mapped_query_embedding,
    testrun=testrun,
)

In [ ]:
adata_ref_latent

In [ ]:
adata_emb = mapping.create_and_save_mapping_output_adata(
    adata_reference=adata_ref,
    adata_reference_latent=adata_ref_latent,
    adata_query=adata_q,
    path_to_mapped_query_emb=path_to_mapped_query_embedding,
    output_path_joint_emb=path_to_joint_ref_q_latent_object,
    add_umap=True,
    add_leiden=True,
    perform_label_transfer=True,
    k=30,
    verbose=True,
)

In [ ]:
if not os.path.isfile(path_to_joint_ref_q_counts_object):
    print(
        f"Creating joint counts object and writing to {path_to_joint_ref_q_counts_object}..."
    )
    adata_counts = mapping.create_joint_ref_q_counts_object(adata_ref, adata_q)
    adata_counts.write(path_to_joint_ref_q_counts_object)
else:
    print(
        "Not creating joint counts object, as it has already been created in a previous run."
    )

In [ ]:
adata = data_cleanup.add_mapping_info_to_counts_adata(
    adata_counts, adata_emb, celltype_col="celltype"
)

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=["r_or_q", "Group", "celltype_merged"], ncols=3, frameon=False)